# Part 1 — Step 2: YOLOv8 Training

**Why YOLOv8?**  
YOLOv8 (Jocher et al., 2023) is a state-of-the-art single-stage detector pretrained on COCO.
It balances speed and accuracy well and handles dense small-object detection (acne lesions) effectively.
Its anchor-free head and CSP backbone make it a strong modern baseline.

**This notebook:**
1. Converts ACNE04 COCO annotations → YOLOv8 label format
2. Writes a `dataset.yaml` config
3. Fine-tunes `yolov8s.pt` (pretrained on COCO) on ACNE04

**Output:** `outputs/yolov8/acne04/weights/best.pt`

In [ ]:
# Fix working directory so relative paths work in both Colab and locally
import os
from pathlib import Path

notebook_dir = Path('part1_detection')
if Path('/content').exists():  # we're in Colab
    os.chdir('/content/AcneDetection/part1_detection')
else:
    # Locally: run from repo root, adjust to notebook dir
    if Path('part1_detection').exists():
        os.chdir('part1_detection')

print(f'Working directory: {os.getcwd()}')

In [7]:
import json
import shutil
from pathlib import Path

import yaml
from ultralytics import YOLO

In [8]:
DATA_DIR   = Path("../data/acne04")
YOLO_DIR   = Path("../data/acne04_yolo")   # converted data
YAML_PATH  = YOLO_DIR / "dataset.yaml"
OUT_DIR    = Path("../outputs/yolov8")

MODEL_NAME = "yolov8s.pt"
EPOCHS     = 50
IMG_SIZE   = 640
BATCH      = 16

SPLITS = ["train", "valid", "test"]

## 1. Convert COCO → YOLOv8 format

YOLOv8 expects one `.txt` label file per image where each row is:
```
class_idx  cx  cy  w  h
```
All values normalised to `[0, 1]` relative to image dimensions.  
COCO stores boxes as `[x_topleft, y_topleft, width, height]` in pixels — so we convert.

In [9]:
def coco_to_yolo(bbox, img_w, img_h):
    """[x, y, w, h] pixels → [cx, cy, w, h] normalised."""
    x, y, w, h = bbox
    return (x + w/2) / img_w, (y + h/2) / img_h, w / img_w, h / img_h

def convert_split(split, cat_to_idx):
    ann_path  = DATA_DIR / split / "_annotations.coco.json"
    yolo_split = "val" if split == "valid" else split
    img_out = YOLO_DIR / "images" / yolo_split
    lbl_out = YOLO_DIR / "labels" / yolo_split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    with open(ann_path) as f:
        coco = json.load(f)

    ann_map = {}
    for ann in coco["annotations"]:
        ann_map.setdefault(ann["image_id"], []).append(ann)

    for meta in coco["images"]:
        src = DATA_DIR / split / meta["file_name"]
        dst = img_out / meta["file_name"]
        if not dst.exists():
            shutil.copy2(src, dst)

        lbl = lbl_out / (Path(meta["file_name"]).stem + ".txt")
        with open(lbl, "w") as f:
            for ann in ann_map.get(meta["id"], []):
                cx, cy, nw, nh = coco_to_yolo(ann["bbox"], meta["width"], meta["height"])
                f.write(f"{cat_to_idx[ann['category_id']]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")

    print(f"  [{split}] done — {len(coco['images'])} images")

In [10]:
if YAML_PATH.exists():
    print("YOLOv8 format already exists, skipping conversion.")
else:
    print("Converting COCO → YOLOv8 format...")
    with open(DATA_DIR / "train" / "_annotations.coco.json") as f:
        coco = json.load(f)

    categories  = sorted(coco["categories"], key=lambda c: c["id"])
    class_names = [c["name"] for c in categories]
    cat_to_idx  = {c["id"]: i for i, c in enumerate(categories)}

    for split in SPLITS:
        convert_split(split, cat_to_idx)

    cfg = {
        "path":  str(YOLO_DIR.resolve()),
        "train": "images/train",
        "val":   "images/val",
        "test":  "images/test",
        "nc":    len(class_names),
        "names": class_names,
    }
    with open(YAML_PATH, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)

    print(f"\ndataset.yaml written → {YAML_PATH}")
    print(f"Classes: {class_names}")

Converting COCO → YOLOv8 format...
  [train] done — 991 images
  [valid] done — 283 images
  [test] done — 142 images

dataset.yaml written → ../data/acne04_yolo/dataset.yaml
Classes: ['Acne', 'nodules and cysts', 'papules', 'pustules', 'whitehead and blackhead']


## 2. Train YOLOv8

In [11]:
model = YOLO(MODEL_NAME)   # downloads yolov8s.pt pretrained weights automatically

results = model.train(
    data     = str(YAML_PATH),
    epochs   = EPOCHS,
    imgsz    = IMG_SIZE,
    batch    = BATCH,
    project  = str(OUT_DIR),
    name     = "acne04",
    exist_ok = True,
    verbose  = True,
)

Ultralytics 8.4.47 🚀 Python-3.13.5 torch-2.10.0 CPU (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/acne04_yolo/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=acne04, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, 

/opt/anaconda3/lib/python3.13/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/06 16:20:44 INFO mlflow.tracking.fluent: Experiment with name '../outputs/yolov8' does not exist. Creating a new experiment.


MLflow: logging run_id(3c6ea84008cd4a4ea1478fa1174714f8) to runs/mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs/mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /Users/evanlee/Desktop/AcneDetection/part1_detection/runs/outputs/yolov8/acne04
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50         0G      3.967      6.636      1.982        357        640: 6% ╸─────────── 4/62 16.2s/it 1:38<15:41:34


KeyboardInterrupt: 

## 3. Check training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

csv_path = OUT_DIR / "acne04" / "results.csv"
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df.plot(x="epoch", y=["train/box_loss", "val/box_loss"], ax=axes[0], title="Box Loss")
df.plot(x="epoch", y=["metrics/precision(B)", "metrics/recall(B)"], ax=axes[1], title="Precision / Recall")
df.plot(x="epoch", y="metrics/mAP50(B)", ax=axes[2], title="mAP@50")
plt.tight_layout()
plt.savefig("../outputs/figures/yolov8_training_curves.png", dpi=150)
plt.show()

best_epoch = df["metrics/mAP50(B)"].idxmax()
print(f"Best epoch : {best_epoch}")
print(f"mAP@50     : {df.loc[best_epoch, 'metrics/mAP50(B)']:.4f}")
print(f"Precision  : {df.loc[best_epoch, 'metrics/precision(B)']:.4f}")
print(f"Recall     : {df.loc[best_epoch, 'metrics/recall(B)']:.4f}")